In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/suedaetiz/super-sms-dataset/super_sms_dataset.csv", encoding="latin-1")
print(df.shape)
df.head()

In [ ]:
# 1. Sütun adlarını sadeleştir
df = df.rename(columns={"SMSes": "text", "Labels": "label"})

# 2. Eksik değerleri kontrol et
print("Eksik değerler:")
print(df.isna().sum())

# 3. Eksik satırları sil
df = df.dropna()

# 4. Etiketleri tam sayıya çevir (1.0 -> 1, 0.0 -> 0)
df["label"] = df["label"].astype(int)

# 5. Tekrar eden mesajları sil
once = len(df)
df = df.drop_duplicates(subset="text")
print("\nSilinen tekrar eden mesaj:", once - len(df))

print("\nTemiz veri boyutu:", df.shape)

In [ ]:
# Mesajların başındaki ve sonundaki boşlukları sil
df["text"] = df["text"].astype(str).str.strip()

# Tamamen boş kalan mesajları sil
df = df[df["text"].str.len() > 0]

# Tekrarları yeniden kontrol et
once = len(df)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print("Silinen tekrar eden mesaj:", once - len(df))
print("Son veri boyutu:", df.shape)

In [ ]:
import matplotlib.pyplot as plt

# Sayılar ve oranlar
sayilar = df["label"].value_counts().sort_index()
print("Normal (0):", sayilar[0], f"(%{sayilar[0] / len(df) * 100:.2f})")
print("Spam   (1):", sayilar[1], f"(%{sayilar[1] / len(df) * 100:.2f})")

# Grafik
plt.figure(figsize=(5, 4))
plt.bar(["Normal", "Spam"], sayilar.values, color=["#4C72B0", "#DD8452"])
for i, v in enumerate(sayilar.values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.ylabel("Mesaj sayısı")
plt.title("Sınıf dağılımı")
plt.tight_layout()
plt.savefig("sinif_dagilimi.png", dpi=150)
plt.show()


In [ ]:
# Her mesajın karakter ve kelime sayısı
df["karakter"] = df["text"].str.len()
df["kelime"] = df["text"].str.split().str.len()

# Sınıflara göre ortalama ve ortanca
print(df.groupby("label")[["karakter", "kelime"]].agg(["mean", "median"]).round(1))

# Grafik: kelime sayısı dağılımı
plt.figure(figsize=(7, 4))
for etiket, ad, renk in [(0, "Normal", "#4C72B0"), (1, "Spam", "#DD8452")]:
    veri = df.loc[df["label"] == etiket, "kelime"]
    plt.hist(veri.clip(upper=80), bins=40, alpha=0.6, label=ad, color=renk)
plt.xlabel("Kelime sayısı")
plt.ylabel("Mesaj sayısı")
plt.title("Mesaj uzunluğu dağılımı")
plt.legend()
plt.tight_layout()
plt.savefig("mesaj_uzunlugu.png", dpi=150)
plt.show()

print("\nMesajların %95'i en fazla kaç kelime?", int(df["kelime"].quantile(0.95)))

In [ ]:
import re
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def kelimelere_ayir(metin):
    kelimeler = re.findall(r"[a-z]+", metin.lower())
    return [k for k in kelimeler if k not in ENGLISH_STOP_WORDS and len(k) > 2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, etiket, ad, renk in [(axes[0], 0, "Normal", "#4C72B0"), (axes[1], 1, "Spam", "#DD8452")]:
    sayac = Counter()
    for metin in df.loc[df["label"] == etiket, "text"]:
        sayac.update(kelimelere_ayir(metin))
    kelimeler, frekanslar = zip(*sayac.most_common(15))
    ax.barh(kelimeler[::-1], frekanslar[::-1], color=renk)
    ax.set_title(f"{ad} mesajlarda en sık 15 kelime")
    ax.set_xlabel("Frekans")
plt.tight_layout()
plt.savefig("sik_kelimeler.png", dpi=150)
plt.show()

In [10]:
once = len(df)
df["text_lower"] = df["text"].str.lower()
df = df.drop_duplicates(subset="text_lower").drop(columns="text_lower").reset_index(drop=True)
print("Büyük/küçük harf farkıyla silinen:", once - len(df))
print("Son boyut:", df.shape)

Büyük/küçük harf farkıyla silinen: 2
Son boyut: (66984, 4)
